In [1]:
import boto3
import pandas as pd
import json
import os
from datetime import datetime, timezone

In [2]:
# LocalStack connection setup
local_endpoint = "http://localhost:4566"
iam_client = boto3.client(
    'iam', 
    endpoint_url=local_endpoint)

print("Connection Successful! LocalStack IAM Service is Active.")

Connection Successful! LocalStack IAM Service is Active.


In [3]:
#AWS IAM response analysis
response = iam_client.list_users()

print(response)

{'Users': [{'Path': '/', 'UserName': 'jane.doe', 'UserId': 'fioq6kr8fkns5txfj96u', 'Arn': 'arn:aws:iam::000000000000:user/jane.doe', 'CreateDate': datetime.datetime(2026, 6, 8, 14, 48, 11, 678000, tzinfo=tzutc())}, {'Path': '/', 'UserName': 'alan.smithee', 'UserId': '964vufm0x94x7c15wl70', 'Arn': 'arn:aws:iam::000000000000:user/alan.smithee', 'CreateDate': datetime.datetime(2026, 6, 8, 14, 48, 11, 730000, tzinfo=tzutc())}, {'Path': '/', 'UserName': 'j.random.hacker', 'UserId': 'r21yyrcmachnf3eq2wxq', 'Arn': 'arn:aws:iam::000000000000:user/j.random.hacker', 'CreateDate': datetime.datetime(2026, 6, 8, 14, 48, 11, 781000, tzinfo=tzutc())}, {'Path': '/', 'UserName': 'joe.shmoe', 'UserId': '3xkwsj3qej9sg7j43crv', 'Arn': 'arn:aws:iam::000000000000:user/joe.shmoe', 'CreateDate': datetime.datetime(2026, 6, 8, 14, 48, 11, 829000, tzinfo=tzutc())}, {'Path': '/', 'UserName': 'alice.smith', 'UserId': 'mhseumthid6f952zxg13', 'Arn': 'arn:aws:iam::000000000000:user/alice.smith', 'CreateDate': datetim

In [4]:
#Audit Timestamp.The date the report was received serves as evidence.
audit_timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
file_suffix = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

In [5]:
inventory_list = []

# Inventory Mapping
for user in response['Users']:
    username = user['UserName']

    #SOC Analysis: listing MFA devices of users
    mfa_response = iam_client.list_mfa_devices(UserName=username)
    mfa_devices = mfa_response.get('MFADevices', [])

    #If there is one device at least, MFA status is active.
    mfa_active = "True" if len(mfa_devices) > 0 else "False"
    
    inventory_list.append({
        "Audit_Date": audit_timestamp,
        "Entity_Type": "IAM_User",
        "Username": username,
        "User_Id": user['UserId'],
        "Arn": user['Arn'],
        "Path": user['Path'],
        "Created_At": user['CreateDate'].strftime('%Y-%m-%d %H:%M:%S UTC'),
        "MFA_Active": mfa_active,
        "Status": "Active" 
    })

In [6]:
#DataFrame Transformation
df_inventory = pd.DataFrame(inventory_list)

In [7]:
#Presentation to the Auditor (Table Display on Screen)
print(f"=== AWS IAM IDENTITY INVENTORY REPORT ===")
print(f"Audit Executed At: {audit_timestamp}")
print(f"Total Identities Found: {len(df_inventory)}\n")
display(df_inventory)

=== AWS IAM IDENTITY INVENTORY REPORT ===
Audit Executed At: 2026-06-08 22:09:41 UTC
Total Identities Found: 6



,Audit_Date,Entity_Type,Username,User_Id,Arn,Path,Created_At,MFA_Active,Status
0,2026-06-08 22:09:41 UTC,IAM_User,jane.doe,fioq6kr8fkns5txfj96u,arn:aws:iam::000000000000:user/jane.doe,/,2026-06-08 14:48:11 UTC,False,Active
1,2026-06-08 22:09:41 UTC,IAM_User,alan.smithee,964vufm0x94x7c15wl70,arn:aws:iam::000000000000:user/alan.smithee,/,2026-06-08 14:48:11 UTC,False,Active
2,2026-06-08 22:09:41 UTC,IAM_User,j.random.hacker,r21yyrcmachnf3eq2wxq,arn:aws:iam::000000000000:user/j.random.hacker,/,2026-06-08 14:48:11 UTC,False,Active
3,2026-06-08 22:09:41 UTC,IAM_User,joe.shmoe,3xkwsj3qej9sg7j43crv,arn:aws:iam::000000000000:user/joe.shmoe,/,2026-06-08 14:48:11 UTC,False,Active
4,2026-06-08 22:09:41 UTC,IAM_User,alice.smith,mhseumthid6f952zxg13,arn:aws:iam::000000000000:user/alice.smith,/,2026-06-08 14:48:11 UTC,False,Active
5,2026-06-08 22:09:41 UTC,IAM_User,bob.jones,d58n3mircpuv9g23ndg5,arn:aws:iam::000000000000:user/bob.jones,/,2026-06-08 14:48:11 UTC,False,Active


In [8]:
#Saving Outputs (For Reporting Folder Structure)
#Creating reports folder and saving them
os.makedirs("../reports", exist_ok=True)

csv_filename = f"iam_inventory_{file_suffix}.csv"
json_filename = f"iam_inventory_{file_suffix}.json"

csv_filepath = os.path.join("../reports", csv_filename)
json_filepath = os.path.join("../reports", json_filename)

#CSV format for Auditor/Excel Compatible
df_inventory.to_csv(csv_filepath, index=False)

# JSON format for SIEM systems/SOC Compatible
with open(json_filepath, 'w') as f:
    json.dump(inventory_list, f, indent=4)

print(f"✅ Reports generated successfully:\n - {csv_filepath} (Auditor/Excel Compatible)\n - {json_filepath} (SIEM/SOC Compatible)")
print("IAM Inventory Report Created Successfully.")

✅ Reports generated successfully:
 - ../reports\iam_inventory_20260608_220941.csv (Auditor/Excel Compatible)
 - ../reports\iam_inventory_20260608_220941.json (SIEM/SOC Compatible)
IAM Inventory Report Created Successfully.
